From: https://mixtape.scunning.com/08-panel_data

In [1]:
import numpy as np 
import pandas as pd 
import statsmodels.api as sm 
import statsmodels.formula.api as smf 
from itertools import combinations 

from tabulate import tabulate
import textwrap

Context: Labor market survey of sex workers.
* Static section: contains worker-specific characteristics
* Panel secetion: information about their last 4 sessions with clients

Estimate the effect of certain acts or client characteristics on log wages. 
Below we will consider a few effects in particular:
* Do clients pay a premium for unprotected sex?
* Do regulars pay less for sessions with a sex worker?
* Are there "volume discounts" in sex work (i.e. do clients pay less per hour for longer sessions)?

In [2]:
url = 'https://github.com/scunning1975/mixtape/raw/master/sasp_panel.dta'
df = pd.read_stata(url)

# drop rows where index is missing
drop_bool = df['id'].isna()
drop_idx = df.loc[drop_bool].index
df = df.drop(index=drop_idx)

In [3]:
df.sort_values(by='id', inplace=True)

In [4]:
df.columns

Index(['id', 'session', 'age', 'age_cl', 'appearance_cl', 'bmi', 'schooling',
       'asq_cl', 'provider_second', 'asian_cl', 'black_cl', 'hispanic_cl',
       'othrace_cl', 'reg', 'hot', 'massage_cl', 'lnw', 'llength', 'unsafe',
       'asian', 'black', 'hispanic', 'other', 'white', 'asq', 'cohab',
       'married', 'divorced', 'separated', 'nevermarried', 'widowed'],
      dtype='object')

In [5]:
df.describe()

,id,session,age,age_cl,appearance_cl,bmi,schooling,asq_cl,asian_cl,black_cl,...,hispanic,other,white,asq,cohab,married,divorced,separated,nevermarried,widowed
count,1727.000000,1727.000000,1701.000000,1645.000000,1676.000000,1684.000000,1724.000000,1645.000000,1661.000000,1661.000000,...,1727.00000,1727.000000,1727.000000,1701.000000,1723.000000,1723.000000,1723.000000,1723.000000,1723.000000,1723.000000
mean,334.852936,2.389693,36.387419,44.989362,6.011337,24.006695,14.584687,2139.727295,0.050572,0.051174,...,0.03011,0.072380,0.833816,1410.432129,0.172374,0.171793,0.287870,0.051654,0.296576,0.019733
std,199.965424,1.109512,9.297238,10.758945,2.179249,5.797507,1.488715,1013.658508,0.219188,0.220419,...,0.17094,0.259191,0.372354,709.613159,0.377815,0.377310,0.452902,0.221392,0.456880,0.139122
min,1.000000,1.000000,18.000000,18.000000,1.000000,15.660503,11.000000,324.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,324.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,162.000000,1.000000,29.000000,37.000000,5.000000,20.482830,14.000000,1369.000000,0.000000,0.000000,...,0.00000,0.000000,1.000000,841.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,314.000000,2.000000,36.000000,45.000000,6.000000,22.148708,14.000000,2025.000000,0.000000,0.000000,...,0.00000,0.000000,1.000000,1296.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,512.000000,3.000000,43.000000,52.500000,8.000000,25.790533,16.000000,2756.250000,0.000000,0.000000,...,0.00000,0.000000,1.000000,1849.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000
max,690.000000,4.000000,65.000000,87.000000,10.000000,54.731834,16.000000,7569.000000,1.000000,1.000000,...,1.00000,1.000000,1.000000,4225.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [6]:
df = df.loc[df.notna().all(axis=1)]

## Generate balanced dataframe

In [7]:
# check there are no duplicates in id and session
assert ~(df[['id', 'session']].duplicated().any()) 
# count the number of sessions per id
session_counts = df.groupby('id')['session'].count()
# get ids that are in all sessions (have 4 unique sessions)
max_number_sessions = len(df['session'].unique())
in_all_sessions = session_counts.loc[session_counts == max_number_sessions].index

# keep only the ids that are in all samples
bal_df = df.loc[df['id'].isin(in_all_sessions)].copy()

bal_df.shape

(1028, 31)

In [8]:
print(bal_df['provider_second'].value_counts())

bal_df['provider_second'] = (bal_df['provider_second'] == '2. Yes') * 1

provider_second
1. No     991
2. Yes     37
Name: count, dtype: int64


In [9]:
features = bal_df.columns.to_list()
features = [x for x in features if x not in ['session', 'id']]

count_within_group_changes = bal_df.groupby('id')[features].nunique()
const_feat = (count_within_group_changes == 1).all(axis=0)
print(const_feat)
const_feat = list(const_feat.loc[const_feat == True].index)

age                 True
age_cl             False
appearance_cl      False
bmi                 True
schooling           True
asq_cl             False
provider_second    False
asian_cl           False
black_cl           False
hispanic_cl        False
othrace_cl         False
reg                False
hot                False
massage_cl         False
lnw                False
llength            False
unsafe             False
asian               True
black               True
hispanic            True
other               True
white               True
asq                 True
cohab               True
married             True
divorced            True
separated           True
nevermarried        True
widowed             True
dtype: bool


In [10]:
bal_df.loc[bal_df['id'] == 6]
bal_df['age_cl'].mean()

np.float64(44.80593385214008)

In [11]:
features = bal_df.columns.to_list()
features = [x for x in features if x not in ['session', 'id']]

# demean_features

# these are non-constat features we want to demean
demean_features = [x for x in features if x not in const_feat]

bal_df.groupby('id')[demean_features].apply(lambda x: x - np.mean(x))

demean_df = bal_df[demean_features] - bal_df.groupby('id')[demean_features].transform('mean')

mean_df = bal_df.groupby('id')[demean_features].transform('mean')
demean_df = bal_df[demean_features] - mean_df
rename_col_list = [col if col == 'id' else f'demean_{col}' for col in demean_df.columns]
demean_df.columns = rename_col_list
# add ID and session to demean df
demean_df = pd.concat([bal_df[['id', 'session']], demean_df], axis=1)

# merge demeaned values into balanced dataset
bal_df = pd.merge(
    left=bal_df,
    right=demean_df,
    on=['id', 'session'],
    how='outer',
    validate='1:1'
)

In [12]:
bal_df.loc[bal_df['id'] == 6, ['id', 'session', 'age', 'age_cl', 'demean_age_cl']]

,id,session,age,age_cl,demean_age_cl
0,6.0,1.0,29.0,32.5,0.375
1,6.0,2.0,29.0,30.0,-2.125
2,6.0,3.0,29.0,45.0,12.875
3,6.0,4.0,29.0,21.0,-11.125


## Pooled OLS

In [13]:
pols_exclude = ['id', 'session', 'lnw'] + list(demean_df.columns)

exog_vars = [col for col in bal_df.columns if col not in pols_exclude]
reg_str = ' + '.join(exog_vars)

reg_str = f'lnw ~ {reg_str}'

print(textwrap.fill(reg_str, 80))

lnw ~ age + age_cl + appearance_cl + bmi + schooling + asq_cl + provider_second
+ asian_cl + black_cl + hispanic_cl + othrace_cl + reg + hot + massage_cl +
llength + unsafe + asian + black + hispanic + other + white + asq + cohab +
married + divorced + separated + nevermarried + widowed


In [14]:
reg_df = bal_df.loc[bal_df[['lnw'] + exog_vars].notna().all(axis=1)].copy()

In [15]:
pols_mod = smf.ols(reg_str, data=reg_df)
pols_res = pols_mod.fit()
pols_res.summary(slim=True)

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    lnw   R-squared:                       0.303
Model:                            OLS   Adj. R-squared:                  0.285
No. Observations:                1028   F-statistic:                     16.72
Covariance Type:            nonrobust   Prob (F-statistic):           1.58e-61
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept           5.1245      0.230     22.284      0.000       4.673       5.576
age                 0.0025      0.012      0.209      0.834      -0.021       0.025
age_cl             -0.0013      0.009     -0.156      0.876      -0.018       0.016
appearance_cl       0.0200      0.007      2.988      0.003       0.007       0.033
bmi                -0.0217      0.002     -9.286      0.000      -0.026      -0.017
schooling           0.0203      0.010      2.025      0.043       0.001       0.040
asq_cl           4.417e-05   9.13e-05      0.484      0.629      -0.000       0.000
provider_second     0.0551      0.072      0.761      0.447      -0.087       0.197
asian_cl           -0.0129      0.059     -0.220      0.826      -0.128       0.102
black_cl            0.0924      0.063      1.472      0.141      -0.031       0.216
hispanic_cl         0.0523      0.076      0.685      0.493      -0.098       0.202
othrace_cl          0.1568      0.081      1.938      0.053      -0.002       0.315
reg                -0.0470      0.028     -1.657      0.098      -0.103       0.009
hot                 0.1329      0.028      4.711      0.000       0.078       0.188
massage_cl         -0.1336      0.029     -4.574      0.000      -0.191      -0.076
llength            -0.3083      0.020    -15.513      0.000      -0.347      -0.269
unsafe              0.0140      0.028      0.493      0.622      -0.042       0.070
asian               1.1560      0.137      8.426      0.000       0.887       1.425
black               1.0983      0.081     13.482      0.000       0.938       1.258
hispanic            0.8437      0.090      9.345      0.000       0.667       1.021
other               0.9578      0.075     12.855      0.000       0.812       1.104
white               1.0688      0.062     17.239      0.000       0.947       1.190
asq                -0.0001      0.000     -0.809      0.419      -0.000       0.000
cohab               0.8157      0.050     16.405      0.000       0.718       0.913
married             0.8746      0.053     16.396      0.000       0.770       0.979
divorced            0.8494      0.053     16.113      0.000       0.746       0.953
separated           0.8144      0.061     13.340      0.000       0.695       0.934
nevermarried        0.8672      0.047     18.319      0.000       0.774       0.960
widowed             0.9031      0.092      9.802      0.000       0.722       1.084
===================================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 3.23e-23. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [16]:
# Function to generate significance stars
def get_significance_stars(pval):
    if pval < 0.001:
        return '****'
    elif pval < 0.01:
        return '***'
    elif pval < 0.05:
        return '**'
    elif pval < 0.1:
        return '*'
    else:
        return ''

pols_res.pvalues.apply(get_significance_stars)

Intercept          ****
age                    
age_cl                 
appearance_cl       ***
bmi                ****
schooling            **
asq_cl                 
provider_second        
asian_cl               
black_cl               
hispanic_cl            
othrace_cl            *
reg                   *
hot                ****
massage_cl         ****
llength            ****
unsafe                 
asian              ****
black              ****
hispanic           ****
other              ****
white              ****
asq                    
cohab              ****
married            ****
divorced           ****
separated          ****
nevermarried       ****
widowed            ****
dtype: object

In [17]:
# combining the above into a function
def get_result_array(res, keep_vars, name=None):
    coef_series = (
        res.params[keep_vars].round(3).astype(str)           # format the estimated parameters as rounded strings
        +                                                        # use addition to add stars (if any)
        res.pvalues.apply(get_significance_stars)[keep_vars] # generate significance stars
    )
    coef_series.name = 'coef'  # naming series will be helpful when this is a dataframe
    tvalue_series = '(' + res.bse[keep_vars].round(3).astype(str) + ')' # add parentheses around formatted t-values
    tvalue_series.name = 'se'
    obs_series = pd.Series([f'{res.nobs:.0f}'], index=['_nobs'], name='obs')

    ols_res_series = pd.concat([coef_series, tvalue_series, obs_series]).sort_index(ascending=False)
    if name is not None:
        ols_res_series.name = name

    # reorder index
    index_order = keep_vars + ['_nobs']
    ols_res_series = ols_res_series.loc[index_order]
    
    return ols_res_series

In [18]:
pols_col = get_result_array(pols_res, keep_vars=['unsafe', 'llength', 'reg'])
pols_col.name = 'POLS'
pols_col

unsafe          0.014
unsafe        (0.028)
llength    -0.308****
llength        (0.02)
reg           -0.047*
reg           (0.028)
_nobs            1028
Name: POLS, dtype: object

## Fixed effects

In [19]:
pols_exclude = ['id', 'session', 'lnw'] + list(demean_df.columns) + const_feat

exog_vars = [col for col in bal_df.columns if col not in pols_exclude]
reg_str = ' + '.join(exog_vars)

reg_str = f'lnw ~ C(id) + {reg_str}'

print(textwrap.fill(reg_str, 80))

# reg_df = bal_df.loc[bal_df[exog_vars].notna().all(axis=1)].copy()
reg_df = bal_df[['id', 'lnw'] + exog_vars].copy()

lnw ~ C(id) + age_cl + appearance_cl + asq_cl + provider_second + asian_cl +
black_cl + hispanic_cl + othrace_cl + reg + hot + massage_cl + llength + unsafe


In [20]:
fe_mod = smf.ols(reg_str, data=reg_df)
fe_res = fe_mod.fit(cov_type='cluster', cov_kwds={'groups': reg_df['id']})
# fe_fit = fe_mod.fit()

# fe_fit.summary()
fe_col = get_result_array(fe_res, keep_vars=['unsafe', 'llength', 'reg'])
fe_col.name = 'FE'
fe_col

unsafe          0.051
unsafe        (0.033)
llength    -0.435****
llength       (0.028)
reg           -0.037*
reg           (0.022)
_nobs            1028
Name: FE, dtype: object

## De-meaned

In [21]:
# cols_exclude =  ['demean_lnw', 'demean_white', 'demean_nevermarried', 'demean_widowed']
cols_exclude = ['demean_' + col for col in const_feat]
cols_exclude = cols_exclude + ['demean_lnw']
# cols_exclude = ['demean_lnw'] + cols_exclude
exog_vars = [col for col in demean_df.columns if col not in cols_exclude]
reg_str = ' + '.join(exog_vars)

reg_str = f'demean_lnw ~ {reg_str}'

print(textwrap.fill(reg_str, 80))

reg_df = bal_df[exog_vars + ['id', 'demean_lnw']].copy()
# reg_df = bal_df.loc[bal_df[exog_vars].notna().all(axis=1)].copy()

demean_lnw ~ id + session + demean_age_cl + demean_appearance_cl + demean_asq_cl
+ demean_provider_second + demean_asian_cl + demean_black_cl +
demean_hispanic_cl + demean_othrace_cl + demean_reg + demean_hot +
demean_massage_cl + demean_llength + demean_unsafe


In [22]:
demean_mod = smf.ols(reg_str, data=reg_df)
demean_res = demean_mod.fit(cov_type='cluster', cov_kwds={'groups': reg_df['id']})
# fe_fit = fe_mod.fit()

demean_res.summary()
demean_col = get_result_array(demean_res, keep_vars=['demean_unsafe', 'demean_llength', 'demean_reg'])
print(demean_col)

demean_col.index = [idx.replace('demean_', '') for idx in demean_col.index]
demean_col.name = 'De-mean'

demean_unsafe          0.045
demean_unsafe        (0.028)
demean_llength    -0.434****
demean_llength       (0.024)
demean_reg          -0.041**
demean_reg           (0.019)
_nobs                   1028
dtype: object


/opt/anaconda3/envs/venv_econ726/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 16, but rank is 14
  warnings.warn('covariance of constraints does not have full '


In [23]:
result_df = pd.concat([pols_col, fe_col, demean_col], axis=1)

# # rename index values
result_df.rename(index={
    'format_ol': 'Dummy online = 1',
    'format_blended': 'Dummy blended = 1',
    '_nobs': 'Observations'}, inplace=True)

print(tabulate(result_df, headers='keys'))



              POLS        FE          De-mean
------------  ----------  ----------  ----------
unsafe        0.014       0.051       0.045
unsafe        (0.028)     (0.033)     (0.028)
llength       -0.308****  -0.435****  -0.434****
llength       (0.02)      (0.028)     (0.024)
reg           -0.047*     -0.037*     -0.041**
reg           (0.028)     (0.022)     (0.019)
Observations  1028        1028        1028
